# Data Transformation

In [1]:
import os
%pwd

'/Users/kiranprasadjp/Documents/Pros/NLP/hf-summarizer/research'

In [2]:
os.chdir("../")
%pwd

'/Users/kiranprasadjp/Documents/Pros/NLP/hf-summarizer'

## Basic Cofiguration

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    rootDir: Path
    dataPath: Path
    tokenizerName: Path



In [4]:
from src.text_summarizer.constants import *
from src.text_summarizer.utils.common import readYaml, createDir

In [5]:
class ConfigurationManager:
    def __init__(self, config_path=CONFIG_FILE_PATH, params_path= PARAMS_FILE_PATH):
        self.config=readYaml(config_path)
        self.params=readYaml(params_path)

        createDir([self.config.artifactsRoot])

    def getDataTransformationConfig(self)-> DataTransformationConfig:
        config= self.config.dataTransformation

        createDir([config.rootDir])

        dataTransformConfig= DataTransformationConfig(
            rootDir= config.rootDir,
            dataPath= config.dataPath,
            tokenizerName= config.tokenizerName,
        )

        return dataTransformConfig

In [6]:
import os
from src.text_summarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_from_disk

/Users/kiranprasadjp/Documents/Pros/NLP/hf-summarizer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Data Transformation Component

In [11]:
class DataTransformation:
    def __init__(self, config:DataTransformationConfig):
        self.config=config
        self.tokenizer=AutoTokenizer.from_pretrained(config.tokenizerName)

    def examplesToFunctions(self,exmBatch):
        inputEncoding=self.tokenizer(exmBatch['dialogue'],max_length=1024, truncation= True)

        targetEncodings=self.tokenizer(exmBatch['summary'],max_length=128, truncation= True)
        return {
            'inputIds': inputEncoding['input_ids'],
            'attentionMask': inputEncoding['attention_mask'],
            'labels': targetEncodings['input_ids']

        }
    def convert(self):
        dfSamsum=load_from_disk(self.config.dataPath)
        dfSamsumPt=dfSamsum.map(self.examplesToFunctions,batched=True)
        dfSamsumPt.save_to_disk(os.path.join(self.config.rootDir,"samsum_dataset"))

In [12]:
config= ConfigurationManager()
dataTransformationConfig =config.getDataTransformationConfig()
dataTransformation= DataTransformation(config=dataTransformationConfig)
dataTransformation.convert()

[2026-03-05 03:31:15,391: INFO: common: YAML file:config/config.yaml loaded successfully]
[2026-03-05 03:31:15,393: INFO: common: YAML file:params.yaml loaded successfully]
[2026-03-05 03:31:15,393: INFO: common: Directory Created at: artifacts]
[2026-03-05 03:31:15,393: INFO: common: Directory Created at: artifacts/data_transformation]
[2026-03-05 03:31:15,540: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-05 03:31:15,564: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-03-05 03:31:15,608: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-05 03:31:15,635: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cac

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 112626.49 examples/s]
